# Gaussian HMM — Behavioral Regime Detection (Main Model)
**Capstone Project | Author: Arevik Melikyan | Supervisor: Varazdat Stepanyan**

> **Purpose:** Hidden Markov Model (HMM) is the **main model** in this capstone. Unlike GMM (baseline), HMM explicitly models the *temporal ordering* of user sessions — capturing how users transition between latent behavioral states over time. This directly addresses Research Question 2: *How do short-term session dynamics compare to long-term user preferences?*
>
> **What HMM adds over GMM:**
> - Models the **Markov transition structure** between states (who transitions from browsing → purchase intent?)
> - Provides **state sequence per user** via the Viterbi algorithm
> - Separates **session-driven volatility** (frequent state switches) from **stable preference states** (persistent states)
> - Computes **posterior state probabilities** at every timestep

---
### Notebook Structure
```
0.  Environment & Imports
1.  Configuration
2.  Data Loading  (loads GMM-labelled output or raw sessions)
3.  Sequence Construction  (user-level temporal session ordering)
4.  Preprocessing Pipeline  (same as GMM for fair comparison)
5.  Optimal K Selection  (BIC / AIC / log-likelihood curves)
6.  Final GaussianHMM Fitting
7.  Viterbi Decoding — State Sequences per User
8.  Transition Matrix Analysis
9.  State Profiling & Interpretation
10. GMM vs HMM Comparison  (key for Research Question 2)
11. Short-term vs Long-term Preference Analysis
12. Dimensionality Reduction Visualizations
13. Stability Validation (bootstrap ARI)
14. Saving Outputs & Run Summary
```

## 0. Environment & Imports

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install hmmlearn pandas numpy scikit-learn matplotlib seaborn umap-learn joblib tqdm

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
import json
import itertools
from pathlib import Path
from datetime import datetime
from copy import deepcopy

# ── Core data ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── HMM ───────────────────────────────────────────────────────────────────────
from hmmlearn.hmm import GaussianHMM

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    confusion_matrix,
)
from sklearn.impute import SimpleImputer

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import seaborn as sns

# ── Optional: UMAP ────────────────────────────────────────────────────────────
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("[INFO] umap-learn not installed — UMAP plots will be skipped.")

# ── Utilities ─────────────────────────────────────────────────────────────────
import joblib
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", font_scale=1.1)
PALETTE = ["#4361EE", "#F72585", "#4CC9F0", "#7209B7",
           "#3A0CA3", "#560BAD", "#480CA8", "#B5179E"]
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print(f"Environment ready | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 1. Configuration

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CFG = {
    # ── Data ──────────────────────────────────────────────────────────────────
    # Option A: Load from GMM notebook output (recommended — same data)
    "gmm_output_path": "gmm_outputs/sessions_with_gmm_labels.parquet",
    # Option B: Load raw session data directly (set gmm_output_path=None)
    "raw_data_path": None,
    # Option C: Simulate fresh (set both above to None)
    "use_simulation": True,
    "n_simulated_users": 50_000,
    "n_simulated_sessions": 300_000,

    # ── Features — MUST match GMM notebook exactly for fair comparison ────────
    "features": [
        "session_length",
        "n_unique_items_viewed",
        "revisit_rate",
        "inter_event_time_median",
        "cart_actions",
        "view_depth",
        "session_velocity",
        "category_diversity",
    ],

    # ── Preprocessing (identical to GMM) ──────────────────────────────────────
    "scaler": "robust",
    "log_transform_cols": [
        "session_length",
        "inter_event_time_median",
        "n_unique_items_viewed",
    ],

    # ── Sequence construction ─────────────────────────────────────────────────
    "min_sessions_per_user": 3,    # Drop users with fewer sessions (too short for HMM)
    "max_sessions_per_user": 500,  # Cap very active users (optional, set None to disable)
    "time_col": "session_start",   # Column used to sort sessions chronologically
                                   # Set None if not available (uses existing row order)

    # ── HMM hyperparameter search ─────────────────────────────────────────────
    "k_range": range(2, 9),
    "covariance_types": ["full", "diag", "tied", "spherical"],
    "n_init": 10,                  # Random restarts per (k, cov_type)
    "max_iter": 200,               # EM iterations
    "tol": 1e-4,

    # ── Final model ───────────────────────────────────────────────────────────
    "final_k": None,               # None = auto-select via BIC
    "final_cov_type": "full",      # Full covariance = richest model

    # ── Stability validation ──────────────────────────────────────────────────
    "bootstrap_n": 20,
    "bootstrap_sample_frac": 0.8,

    # ── Silhouette sampling ───────────────────────────────────────────────────
    "silhouette_sample_size": 10_000,

    # ── Output ────────────────────────────────────────────────────────────────
    "output_dir": "hmm_outputs",
}

Path(CFG["output_dir"]).mkdir(parents=True, exist_ok=True)
print("Configuration loaded.")
for k, v in CFG.items():
    print(f"  {k:40s}: {v}")

## 2. Data Loading

In [ ]:
def simulate_c2c_with_temporal(
    n_users: int,
    n_sessions: int,
    random_state: int = 42
) -> pd.DataFrame:
    """
    Simulate C2C sessions with realistic temporal structure.

    4 hidden regimes with Markov-like transition tendencies:
      0 — Casual browser     (tends to stay casual or churn)
      1 — Comparison shopper (tends to escalate to purchase-intent)
      2 — Purchase-intent    (may convert or return to casual)
      3 — Cold-start / churn (tends to stay cold or re-engage)

    The temporal ordering within each user follows a plausible
    Markov chain, making HMM recovery meaningful.
    """
    rng = np.random.default_rng(random_state)

    # Ground-truth transition matrix (rows = from, cols = to)
    TRUE_TRANS = np.array([
        [0.60, 0.25, 0.05, 0.10],  # from casual
        [0.15, 0.50, 0.30, 0.05],  # from comparison
        [0.20, 0.20, 0.50, 0.10],  # from purchase-intent
        [0.25, 0.05, 0.05, 0.65],  # from cold-start
    ])

    regime_params = {
        0: dict(sess_len=(180, 60),   n_items=(4, 2),   revisit=(0.10, 0.05),
                iet=(120, 60),        cart=(0.2, 0.3),  depth=(2.0, 1.0),
                vel=(0.03, 0.01),     cat_div=(1.2, 0.5)),
        1: dict(sess_len=(600, 150),  n_items=(12, 4),  revisit=(0.45, 0.12),
                iet=(45, 20),         cart=(1.2, 0.8),  depth=(5.0, 1.5),
                vel=(0.07, 0.02),     cat_div=(2.5, 0.8)),
        2: dict(sess_len=(1200, 300), n_items=(20, 6),  revisit=(0.65, 0.15),
                iet=(20, 10),         cart=(3.5, 1.2),  depth=(9.0, 2.5),
                vel=(0.12, 0.03),     cat_div=(1.8, 0.6)),
        3: dict(sess_len=(40, 20),    n_items=(1, 1),   revisit=(0.02, 0.02),
                iet=(600, 200),       cart=(0.0, 0.1),  depth=(1.0, 0.3),
                vel=(0.01, 0.005),    cat_div=(1.0, 0.2)),
    }

    def _sample(mean, std, size, low=0.0):
        return np.clip(rng.normal(mean, std, size), low, None)

    rows = []
    sessions_per_user = n_sessions // n_users
    # Initial state distribution
    init_probs = np.array([0.35, 0.30, 0.20, 0.15])

    session_counter = 0
    base_time = pd.Timestamp("2023-01-01")

    for uid in range(n_users):
        n_sess = max(3, int(rng.normal(sessions_per_user, sessions_per_user * 0.3)))
        state = rng.choice(4, p=init_probs)
        t = base_time + pd.Timedelta(days=int(rng.uniform(0, 30)))

        for s in range(n_sess):
            p = regime_params[state]
            rows.append({
                "user_id":               uid,
                "session_id":            session_counter,
                "session_start":         t,
                "true_regime":           state,
                "session_length":        float(_sample(*p["sess_len"], 1, 10)),
                "n_unique_items_viewed": int(max(1, round(_sample(*p["n_items"], 1, 1)[0]))),
                "revisit_rate":          float(np.clip(_sample(*p["revisit"], 1), 0, 1)),
                "inter_event_time_median": float(_sample(*p["iet"], 1, 1)),
                "cart_actions":          int(max(0, round(_sample(*p["cart"], 1, 0)[0]))),
                "view_depth":            float(round(_sample(*p["depth"], 1, 1)[0], 1)),
                "session_velocity":      float(np.clip(_sample(*p["vel"], 1), 0.001, None)),
                "category_diversity":    float(np.clip(_sample(*p["cat_div"], 1), 1.0, None)),
            })
            # Markov transition
            state = rng.choice(4, p=TRUE_TRANS[state])
            # Time gap between sessions (30 min to 7 days)
            t += pd.Timedelta(minutes=int(rng.uniform(30, 60 * 24 * 7)))
            session_counter += 1

    df = pd.DataFrame(rows)
    print(f"Simulated: {len(df):,} sessions | {df['user_id'].nunique():,} users")
    print(f"Avg sessions/user: {len(df)/df['user_id'].nunique():.1f}")
    print(f"True regime distribution:\n{df['true_regime'].value_counts().sort_index()}")
    return df


# ── Execute ───────────────────────────────────────────────────────────────────
gmm_path = CFG.get("gmm_output_path")

if gmm_path and Path(gmm_path).exists() and not CFG["use_simulation"]:
    print(f"[MODE] Loading from GMM output: {gmm_path}")
    df_raw = pd.read_parquet(gmm_path)
    HAS_TRUE_LABELS  = "true_regime"  in df_raw.columns
    HAS_GMM_LABELS   = "gmm_regime"   in df_raw.columns
    print(f"Loaded {len(df_raw):,} rows | GMM labels: {HAS_GMM_LABELS}")
elif CFG.get("raw_data_path") and not CFG["use_simulation"]:
    p = Path(CFG["raw_data_path"])
    df_raw = pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p)
    HAS_TRUE_LABELS = "true_regime" in df_raw.columns
    HAS_GMM_LABELS  = "gmm_regime"  in df_raw.columns
    print(f"Loaded {len(df_raw):,} rows")
else:
    print("[MODE] Simulation — generating temporal C2C data...")
    df_raw = simulate_c2c_with_temporal(
        n_users=CFG["n_simulated_users"],
        n_sessions=CFG["n_simulated_sessions"],
        random_state=RANDOM_STATE,
    )
    HAS_TRUE_LABELS = True
    HAS_GMM_LABELS  = False

df_raw.head(3)

## 3. Sequence Construction
> **Critical for HMM.** HMM requires data as a list of observation sequences — one per user, ordered chronologically. GMM receives a flat matrix; HMM receives a list of matrices.

In [ ]:
def build_user_sequences(
    df: pd.DataFrame,
    features: list,
    user_col: str = "user_id",
    time_col: str = None,
    min_sessions: int = 3,
    max_sessions: int = None,
) -> tuple:
    """
    Build per-user session sequences for HMM fitting.

    hmmlearn expects:
        X      : np.ndarray of shape (total_sessions_across_all_users, n_features)
        lengths: list of int — number of sessions per user (must sum to len(X))

    Returns:
        sequences : list of pd.DataFrame — one per user, chronologically ordered
        user_ids  : list of user IDs in the same order
        df_valid  : filtered and ordered DataFrame
    """
    df = df.copy()

    # Sort chronologically within each user
    if time_col and time_col in df.columns:
        df = df.sort_values([user_col, time_col])
    else:
        df = df.sort_values(user_col)  # fallback: assume existing order is temporal

    # Filter by sequence length
    session_counts = df.groupby(user_col).size()
    valid_users = session_counts[
        session_counts >= min_sessions
    ].index

    n_dropped = df[user_col].nunique() - len(valid_users)
    print(f"Dropped {n_dropped:,} users with < {min_sessions} sessions")

    df_valid = df[df[user_col].isin(valid_users)].copy()

    # Cap very active users (optional)
    if max_sessions:
        df_valid = df_valid.groupby(user_col).head(max_sessions)

    sequences = []
    user_ids  = []

    for uid, grp in df_valid.groupby(user_col, sort=False):
        sequences.append(grp)
        user_ids.append(uid)

    lengths = [len(seq) for seq in sequences]
    total   = sum(lengths)

    print(f"Valid users           : {len(sequences):,}")
    print(f"Total sessions        : {total:,}")
    print(f"Avg sessions/user     : {np.mean(lengths):.1f}")
    print(f"Median sessions/user  : {np.median(lengths):.1f}")
    print(f"Max sessions/user     : {max(lengths)}")

    return sequences, user_ids, df_valid


sequences, user_ids, df_valid = build_user_sequences(
    df=df_raw,
    features=CFG["features"],
    user_col="user_id",
    time_col=CFG.get("time_col"),
    min_sessions=CFG["min_sessions_per_user"],
    max_sessions=CFG.get("max_sessions_per_user"),
)

# ── Session length distribution ───────────────────────────────────────────────
lengths_arr = np.array([len(s) for s in sequences])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(lengths_arr, bins=50, color=PALETTE[0], alpha=0.75, edgecolor="none")
axes[0].axvline(np.median(lengths_arr), color="red", ls="--", lw=1.5,
                label=f"Median = {np.median(lengths_arr):.0f}")
axes[0].set_xlabel("Sessions per user")
axes[0].set_ylabel("Users")
axes[0].set_title("User sequence length distribution", fontweight="bold")
axes[0].legend()

axes[1].hist(np.log1p(lengths_arr), bins=50, color=PALETTE[1], alpha=0.75, edgecolor="none")
axes[1].set_xlabel("log(Sessions per user + 1)")
axes[1].set_title("Same — log scale (long-tail check)", fontweight="bold")

plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/01_sequence_lengths.png", bbox_inches="tight")
plt.show()
print("[SAVED] 01_sequence_lengths.png")

## 4. Preprocessing Pipeline
> Identical transforms as GMM for a fair comparison.

In [ ]:
def preprocess_for_hmm(
    sequences: list,
    features: list,
    log_transform_cols: list,
    scaler_type: str = "robust",
) -> tuple:
    """
    Preprocess sequences for HMM:
      1. Stack all sessions into a flat matrix
      2. Impute → log1p → scale
      3. Record lengths array for hmmlearn

    Returns:
        X        : np.ndarray (total_sessions, n_features) — scaled
        lengths  : list[int]
        scaler   : fitted scaler (for inverse_transform later)
    """
    # Stack all user sequences into one matrix (preserve order)
    X_raw = np.vstack([seq[features].values for seq in sequences])
    lengths = [len(seq) for seq in sequences]

    # Impute
    imputer = SimpleImputer(strategy="median")
    X_imp = imputer.fit_transform(X_raw)
    X_df  = pd.DataFrame(X_imp, columns=features)

    # Log-transform
    for col in log_transform_cols:
        if col in features:
            X_df[col] = np.log1p(X_df[col])

    # Scale
    Scaler = RobustScaler if scaler_type == "robust" else StandardScaler
    scaler = Scaler()
    X_scaled = scaler.fit_transform(X_df)

    print(f"Feature matrix   : {X_scaled.shape}")
    print(f"Lengths array    : {len(lengths)} sequences, sum={sum(lengths)}")
    assert sum(lengths) == len(X_scaled), "Length mismatch!"

    return X_scaled, lengths, scaler


X, lengths, fitted_scaler = preprocess_for_hmm(
    sequences=sequences,
    features=CFG["features"],
    log_transform_cols=CFG["log_transform_cols"],
    scaler_type=CFG["scaler"],
)

print(f"\nFeature means (scaled): {X.mean(axis=0).round(3)}")
print(f"Feature stds  (scaled): {X.std(axis=0).round(3)}")

## 5. Optimal K Selection
> HMM model selection uses the same BIC/AIC framework as GMM. The key difference: BIC is computed on the full sequence log-likelihood, accounting for the Markov structure.

In [ ]:
def fit_hmm_safe(
    X: np.ndarray,
    lengths: list,
    k: int,
    cov_type: str,
    n_init: int,
    max_iter: int,
    tol: float,
    random_state: int,
) -> tuple:
    """
    Fit GaussianHMM with multiple random restarts.
    Returns (best_model, best_score) where score = log-likelihood.

    Multiple restarts are essential because EM for HMMs is highly
    sensitive to initialization — a single run often converges to
    a local optimum.
    """
    best_model = None
    best_score = -np.inf

    for i in range(n_init):
        try:
            model = GaussianHMM(
                n_components=k,
                covariance_type=cov_type,
                n_iter=max_iter,
                tol=tol,
                random_state=random_state + i,
                verbose=False,
            )
            model.fit(X, lengths)
            score = model.score(X, lengths)
            if score > best_score:
                best_score = score
                best_model = deepcopy(model)
        except Exception:
            continue  # some initialisations diverge — skip

    return best_model, best_score


def hmm_bic(model: GaussianHMM, X: np.ndarray, lengths: list) -> float:
    """
    Compute BIC for a fitted GaussianHMM.

    BIC = -2 * log-likelihood + n_params * log(n_samples)

    n_params for GaussianHMM (full covariance):
        - Transition matrix    : K*(K-1)  (each row sums to 1)
        - Initial distribution : K-1
        - Emission means       : K * D
        - Emission covariances : K * D*(D+1)/2  (for 'full')
    """
    K = model.n_components
    D = X.shape[1]
    cov_type = model.covariance_type

    if cov_type == "full":
        cov_params = K * D * (D + 1) // 2
    elif cov_type == "diag":
        cov_params = K * D
    elif cov_type == "tied":
        cov_params = D * (D + 1) // 2
    elif cov_type == "spherical":
        cov_params = K
    else:
        cov_params = K * D * (D + 1) // 2

    n_params = (K * (K - 1)) + (K - 1) + (K * D) + cov_params
    log_likelihood = model.score(X, lengths) * len(X)
    n = len(X)
    return -2 * log_likelihood + n_params * np.log(n)


def hmm_model_selection(
    X: np.ndarray,
    lengths: list,
    k_range,
    covariance_types: list,
    n_init: int,
    max_iter: int,
    tol: float,
    silhouette_sample_size: int,
    random_state: int,
) -> pd.DataFrame:
    """
    Grid search over (k, covariance_type) for GaussianHMM.
    Returns a DataFrame of all configurations sorted by BIC.
    """
    results = []
    total = len(list(k_range)) * len(covariance_types)
    pbar = tqdm(total=total, desc="HMM model selection")

    for cov_type in covariance_types:
        for k in k_range:
            model, log_score = fit_hmm_safe(
                X, lengths, k, cov_type, n_init, max_iter, tol, random_state
            )
            if model is None:
                pbar.update(1)
                continue

            bic = hmm_bic(model, X, lengths)
            aic = -2 * (log_score * len(X)) + 2 * (k * (k - 1) + k * X.shape[1])

            labels = model.predict(X, lengths)
            n_unique = len(np.unique(labels))
            if n_unique >= 2:
                sample_idx = np.random.choice(
                    len(X), min(silhouette_sample_size, len(X)), replace=False
                )
                sil = silhouette_score(X[sample_idx], labels[sample_idx])
                db  = davies_bouldin_score(X, labels)
                ch  = calinski_harabasz_score(X, labels)
            else:
                sil, db, ch = np.nan, np.nan, np.nan

            results.append({
                "k":                 k,
                "cov_type":          cov_type,
                "bic":               bic,
                "aic":               aic,
                "log_likelihood":    log_score,
                "silhouette":        sil,
                "davies_bouldin":    db,
                "calinski_harabasz": ch,
                "converged":         model.monitor_.converged,
            })
            pbar.update(1)

    pbar.close()
    return pd.DataFrame(results).sort_values("bic")


print("Running HMM model selection grid search...")
selection_df = hmm_model_selection(
    X=X,
    lengths=lengths,
    k_range=CFG["k_range"],
    covariance_types=CFG["covariance_types"],
    n_init=CFG["n_init"],
    max_iter=CFG["max_iter"],
    tol=CFG["tol"],
    silhouette_sample_size=CFG["silhouette_sample_size"],
    random_state=RANDOM_STATE,
)

print("\nTop 10 configurations by BIC:")
display(selection_df.head(10).reset_index(drop=True))
selection_df.to_csv(f"{CFG['output_dir']}/model_selection_results.csv", index=False)
print("[SAVED] model_selection_results.csv")

In [ ]:
# ── Plot BIC / AIC / Silhouette curves ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = [
    ("bic",       "BIC (lower = better)",              True),
    ("aic",       "AIC (lower = better)",              True),
    ("silhouette","Silhouette score (higher = better)", False),
]

for ax, (metric, title, lower_better) in zip(axes, metrics):
    for j, cov_type in enumerate(CFG["covariance_types"]):
        sub = selection_df[selection_df["cov_type"] == cov_type].sort_values("k")
        ax.plot(sub["k"], sub[metric], marker="o", lw=2,
                color=PALETTE[j], label=cov_type, alpha=0.85)
    best_row = selection_df[
        selection_df["cov_type"] == CFG["final_cov_type"]
    ].sort_values(metric, ascending=lower_better).iloc[0]
    ax.axvline(best_row["k"], color="red", ls="--", lw=1.5,
               label=f"Best k={int(best_row['k'])} (full)")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Number of hidden states (K)")
    ax.set_xticks(list(CFG["k_range"]))
    ax.legend(fontsize=9)

fig.suptitle("HMM model selection metrics", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/02_model_selection_curves.png", bbox_inches="tight")
plt.show()
print("[SAVED] 02_model_selection_curves.png")

In [ ]:
# ── Auto-select K ─────────────────────────────────────────────────────────────
if CFG["final_k"] is None:
    best = selection_df[
        selection_df["cov_type"] == CFG["final_cov_type"]
    ].sort_values("bic").iloc[0]
    BEST_K = int(best["k"])
    print(f"[AUTO] Best K = {BEST_K} "
          f"(BIC={best['bic']:.1f}, silhouette={best['silhouette']:.3f})")
else:
    BEST_K = CFG["final_k"]
    print(f"[MANUAL] Using K = {BEST_K}")

print(f"Final config: K={BEST_K}, covariance_type='{CFG['final_cov_type']}'")

## 6. Final GaussianHMM Fitting

In [ ]:
print(f"Fitting final GaussianHMM: K={BEST_K}, cov_type='{CFG['final_cov_type']}'")
print(f"Using {CFG['n_init'] * 2} random restarts for the final model...")

final_hmm, final_score = fit_hmm_safe(
    X=X,
    lengths=lengths,
    k=BEST_K,
    cov_type=CFG["final_cov_type"],
    n_init=CFG["n_init"] * 2,
    max_iter=CFG["max_iter"],
    tol=CFG["tol"],
    random_state=RANDOM_STATE,
)

assert final_hmm is not None, "HMM fitting failed — check data or reduce K"

print(f"\n── Final Model Summary ─────────────────────────────────")
print(f"  Converged              : {final_hmm.monitor_.converged}")
print(f"  EM iterations          : {final_hmm.monitor_.iter}")
print(f"  Log-likelihood/sample  : {final_score:.4f}")
print(f"  Total log-likelihood   : {final_score * len(X):.1f}")
print(f"  BIC                    : {hmm_bic(final_hmm, X, lengths):.2f}")

print("\nInitial state distribution (π):")
for k, p in enumerate(final_hmm.startprob_):
    print(f"  State {k}: {p:.4f}")

# ── Save model ────────────────────────────────────────────────────────────────
joblib.dump(final_hmm,    f"{CFG['output_dir']}/hmm_final_model.pkl")
joblib.dump(fitted_scaler,f"{CFG['output_dir']}/hmm_scaler.pkl")
print("\n[SAVED] hmm_final_model.pkl + hmm_scaler.pkl")

## 7. Viterbi Decoding — State Sequences per User
> The Viterbi algorithm finds the single most-likely hidden state sequence for each user's observed data. This is what makes HMM uniquely powerful: you get a *trajectory* through states, not just a static label.

In [ ]:
# ── Viterbi decoding ──────────────────────────────────────────────────────────
viterbi_labels = final_hmm.predict(X, lengths)  # flat array, same order as X

# Posterior state probabilities (forward-backward)
posteriors = final_hmm.predict_proba(X, lengths)  # shape: (n_sessions, K)

# ── Attach back to the valid session dataframe ────────────────────────────────
df_valid = df_valid.copy()
df_valid["hmm_state"]     = viterbi_labels
df_valid["max_posterior"] = posteriors.max(axis=1)
for k in range(BEST_K):
    df_valid[f"prob_state_{k}"] = posteriors[:, k]

# ── Per-user state sequences ──────────────────────────────────────────────────
def build_user_state_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate per-user state sequence statistics.

    Returns one row per user with:
        - dominant_state    : most frequent HMM state
        - n_transitions     : total state changes in sequence
        - state_entropy     : Shannon entropy of state distribution (diversity)
        - pct_in_state_k    : fraction of sessions in each state
        - volatility        : transitions / sequence_length (normalized)
        - is_persistent     : True if >70% of sessions in one state
    """
    rows = []
    for uid, grp in df.groupby("user_id"):
        seq = grp["hmm_state"].values
        n   = len(seq)

        transitions = int((seq[1:] != seq[:-1]).sum())
        state_counts = np.bincount(seq, minlength=BEST_K)
        state_fracs  = state_counts / n

        # Shannon entropy of state distribution
        entropy = -np.sum(
            state_fracs * np.log(state_fracs + 1e-10)
        )

        row = {
            "user_id":        uid,
            "n_sessions":     n,
            "dominant_state": int(state_fracs.argmax()),
            "n_transitions":  transitions,
            "volatility":     transitions / max(n - 1, 1),
            "state_entropy":  entropy,
            "is_persistent":  bool(state_fracs.max() > 0.7),
        }
        for k in range(BEST_K):
            row[f"pct_state_{k}"] = state_fracs[k]

        rows.append(row)

    return pd.DataFrame(rows)


df_users = build_user_state_df(df_valid)

print("Per-user state summary:")
display(df_users.describe().round(3))

print(f"\nPersistent users (>70% in one state): "
      f"{df_users['is_persistent'].mean()*100:.1f}%")
print(f"Volatile users (>50% transitions)   : "
      f"{(df_users['volatility'] > 0.5).mean()*100:.1f}%")

In [ ]:
# ── Example state sequence trajectories ──────────────────────────────────────
# Show 6 representative users: 3 persistent, 3 volatile
persistent_users = df_users.sort_values("volatility").head(3)["user_id"].tolist()
volatile_users   = df_users.sort_values("volatility", ascending=False).head(3)["user_id"].tolist()
example_users    = persistent_users + volatile_users
example_labels   = ["Persistent"] * 3 + ["Volatile"] * 3

fig, axes = plt.subplots(2, 3, figsize=(18, 7))
axes = axes.flatten()

state_cmap = {k: PALETTE[k] for k in range(BEST_K)}

for i, (uid, label) in enumerate(zip(example_users, example_labels)):
    ax = axes[i]
    grp = df_valid[df_valid["user_id"] == uid].sort_values(
        CFG["time_col"] if CFG["time_col"] and CFG["time_col"] in df_valid.columns
        else "session_id"
    )
    seq     = grp["hmm_state"].values
    post    = grp["max_posterior"].values
    session_idx = np.arange(len(seq))

    # Plot state as colored step line
    for s in range(len(seq) - 1):
        ax.plot([session_idx[s], session_idx[s+1]],
                [seq[s], seq[s+1]],
                color=state_cmap[seq[s]], lw=2, alpha=0.8)
    ax.scatter(session_idx, seq,
               c=[state_cmap[s] for s in seq], s=50, zorder=5)

    # Posterior confidence as shaded background
    ax2 = ax.twinx()
    ax2.fill_between(session_idx, post, alpha=0.12, color="gray")
    ax2.set_ylim(0, 1.3)
    ax2.set_ylabel("Posterior conf.", fontsize=8, color="gray")
    ax2.tick_params(axis="y", colors="gray", labelsize=7)

    vol = df_users.loc[df_users["user_id"] == uid, "volatility"].values[0]
    ax.set_title(f"{label} User {uid}  |  volatility={vol:.2f}",
                 fontweight="bold", fontsize=10)
    ax.set_xlabel("Session index")
    ax.set_ylabel("Hidden state")
    ax.set_yticks(range(BEST_K))
    ax.set_yticklabels([f"State {k}" for k in range(BEST_K)])
    ax.set_ylim(-0.5, BEST_K - 0.5)

# Legend
patches = [mpatches.Patch(color=PALETTE[k], label=f"State {k}")
           for k in range(BEST_K)]
fig.legend(handles=patches, loc="lower center", ncol=BEST_K,
           bbox_to_anchor=(0.5, -0.02), fontsize=10)

fig.suptitle("Example user state trajectories — persistent vs volatile",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/03_example_trajectories.png", bbox_inches="tight")
plt.show()
print("[SAVED] 03_example_trajectories.png")

## 8. Transition Matrix Analysis
> The transition matrix **A** is the core HMM result. Entry A[i, j] = probability of moving from state i to state j. This encodes the *dynamics* of user behavior — which behaviors escalate toward purchase, which lead to churn.

In [ ]:
# ── Print and visualize the transition matrix ─────────────────────────────────
trans_matrix = final_hmm.transmat_

print("Transition Matrix A (rows = from state, cols = to state):")
trans_df = pd.DataFrame(
    trans_matrix,
    index=[f"From State {k}" for k in range(BEST_K)],
    columns=[f"To State {k}" for k in range(BEST_K)]
).round(4)
display(trans_df)

# ── Transition matrix heatmap ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
ax = axes[0]
sns.heatmap(
    trans_matrix, annot=True, fmt=".3f",
    cmap="Blues", ax=ax,
    xticklabels=[f"State {k}" for k in range(BEST_K)],
    yticklabels=[f"State {k}" for k in range(BEST_K)],
    vmin=0, vmax=1, linewidths=0.5,
    cbar_kws={"label": "Transition probability"}
)
ax.set_xlabel("To state")
ax.set_ylabel("From state")
ax.set_title("Transition matrix A", fontweight="bold", fontsize=12)

# Self-transition (persistence) bar chart
ax2 = axes[1]
self_trans = np.diag(trans_matrix)
bars = ax2.bar(
    [f"State {k}" for k in range(BEST_K)],
    self_trans,
    color=PALETTE[:BEST_K], alpha=0.85, edgecolor="none"
)
ax2.axhline(0.5, color="red", ls="--", lw=1.5, label="50% self-transition")
ax2.set_ylim(0, 1)
ax2.set_ylabel("Self-transition probability P(stay in state)")
ax2.set_title("State persistence (diagonal of A)", fontweight="bold", fontsize=12)
ax2.legend()
for bar, val in zip(bars, self_trans):
    ax2.text(bar.get_x() + bar.get_width() / 2, val + 0.01,
             f"{val:.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/04_transition_matrix.png", bbox_inches="tight")
plt.show()
print("[SAVED] 04_transition_matrix.png")

# ── Stationary distribution ───────────────────────────────────────────────────
# Stationary distribution: left eigenvector of transition matrix
eigenvalues, eigenvectors = np.linalg.eig(trans_matrix.T)
stationary = np.real(eigenvectors[:, np.argmax(np.abs(eigenvalues - 1.0) < 1e-6)])
stationary = np.abs(stationary) / np.abs(stationary).sum()

print("\nStationary distribution (long-run time in each state):")
for k, p in enumerate(stationary):
    print(f"  State {k}: {p:.4f} ({100*p:.1f}%)")

In [ ]:
# ── Mean first passage time (MFPT) ───────────────────────────────────────────
# MFPT[i, j] = expected number of steps to reach state j from state i
# Useful: how many sessions until a casual user becomes purchase-intent?

def mean_first_passage_time(trans: np.ndarray) -> np.ndarray:
    """
    Compute the Mean First Passage Time matrix from a transition matrix.
    Uses the fundamental matrix approach (Kemeny & Snell).
    """
    K = trans.shape[0]
    eigenvalues, eigenvectors = np.linalg.eig(trans.T)
    stationary_idx = np.argmax(np.abs(eigenvalues - 1.0) < 1e-6)
    pi = np.abs(eigenvectors[:, stationary_idx])
    pi /= pi.sum()

    # Fundamental matrix Z = (I - A + PI)^{-1} where PI_ij = pi_j
    PI = np.tile(pi, (K, 1))
    Z  = np.linalg.inv(np.eye(K) - trans + PI)

    mfpt = np.zeros((K, K))
    for i in range(K):
        for j in range(K):
            mfpt[i, j] = (Z[j, j] - Z[i, j]) / pi[j]
    np.fill_diagonal(mfpt, 1 / pi)  # return time

    return mfpt


mfpt = mean_first_passage_time(trans_matrix)
mfpt_df = pd.DataFrame(
    mfpt.round(1),
    index=[f"From State {k}" for k in range(BEST_K)],
    columns=[f"To State {k}" for k in range(BEST_K)]
)

print("Mean First Passage Time (sessions until reaching each state):")
display(mfpt_df)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    mfpt, annot=True, fmt=".1f",
    cmap="YlOrRd", ax=ax,
    xticklabels=[f"State {k}" for k in range(BEST_K)],
    yticklabels=[f"State {k}" for k in range(BEST_K)],
    linewidths=0.5,
    cbar_kws={"label": "Expected sessions"}
)
ax.set_xlabel("Target state")
ax.set_ylabel("Starting state")
ax.set_title("Mean First Passage Time (sessions)", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/05_mean_first_passage_time.png", bbox_inches="tight")
plt.show()
print("[SAVED] 05_mean_first_passage_time.png")

## 9. State Profiling & Interpretation

In [ ]:
# ── Emission means in original feature space ──────────────────────────────────
# HMM emission means are in scaled space — invert to original for interpretability
means_scaled = final_hmm.means_          # shape: (K, D)

# Undo scaling
means_unscaled = fitted_scaler.inverse_transform(means_scaled)

# Undo log-transform for log-transformed features
means_original = means_unscaled.copy()
log_col_idx = [CFG["features"].index(c)
               for c in CFG["log_transform_cols"]
               if c in CFG["features"]]
for idx in log_col_idx:
    means_original[:, idx] = np.expm1(means_unscaled[:, idx])

profile_df = pd.DataFrame(
    means_original,
    columns=CFG["features"],
    index=[f"State {k}" for k in range(BEST_K)]
)
profile_df["n_sessions"] = [
    (viterbi_labels == k).sum() for k in range(BEST_K)
]
profile_df["pct"] = profile_df["n_sessions"] / len(viterbi_labels) * 100

print("State emission means (original feature scale):")
display(profile_df.round(3))
profile_df.to_csv(f"{CFG['output_dir']}/state_profiles.csv")
print("[SAVED] state_profiles.csv")

In [ ]:
# ── Radar chart — state profiles ──────────────────────────────────────────────
profile_norm = profile_df[CFG["features"]].copy()
profile_norm = (profile_norm - profile_norm.min()) / (
    profile_norm.max() - profile_norm.min() + 1e-9
)

n_feat  = len(CFG["features"])
angles  = np.linspace(0, 2 * np.pi, n_feat, endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(1, BEST_K, figsize=(5 * BEST_K, 5),
                         subplot_kw=dict(polar=True))
if BEST_K == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    values = profile_norm.iloc[i].tolist() + [profile_norm.iloc[i].tolist()[0]]
    ax.plot(angles, values, color=PALETTE[i], lw=2)
    ax.fill(angles, values, color=PALETTE[i], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(
        [f.replace("_", "\n") for f in CFG["features"]], fontsize=8
    )
    ax.set_yticklabels([])
    pct = profile_df.iloc[i]["pct"]
    ax.set_title(f"State {i}\n({pct:.1f}% of sessions)",
                 fontsize=11, fontweight="bold", pad=15)

fig.suptitle("HMM emission state profiles (normalized)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/06_state_radar.png", bbox_inches="tight")
plt.show()
print("[SAVED] 06_state_radar.png")

In [ ]:
# ── State label assignment ─────────────────────────────────────────────────────
# Heuristic auto-labeling — always review and override manually
def auto_label_states(profile: pd.DataFrame, features: list) -> dict:
    labels = {}
    cart_col   = "cart_actions"            if "cart_actions"            in features else None
    sess_col   = "session_length"          if "session_length"          in features else None
    revisit_col= "revisit_rate"            if "revisit_rate"            in features else None
    vel_col    = "session_velocity"        if "session_velocity"        in features else None

    for k in profile.index:
        row  = profile.loc[k, features]
        rank = row.rank(pct=True)

        cart_p  = rank[cart_col]    if cart_col    else 0.5
        sess_p  = rank[sess_col]    if sess_col    else 0.5
        rev_p   = rank[revisit_col] if revisit_col else 0.5
        vel_p   = rank[vel_col]     if vel_col     else 0.5

        if cart_p > 0.7 and sess_p > 0.6:
            label = "Purchase-intent"
        elif rev_p > 0.65 and cart_p > 0.4:
            label = "Comparison shopper"
        elif sess_p < 0.3 and cart_p < 0.3 and vel_p < 0.3:
            label = "Cold-start / churn"
        else:
            label = "Casual browser"
        labels[k] = label

    return labels


auto_labels = auto_label_states(profile_df, CFG["features"])
print("Auto-generated state labels (review and override as needed):")
for k, label in auto_labels.items():
    print(f"  State {k} → '{label}'")

# ── OVERRIDE HERE ─────────────────────────────────────────────────────────────
# STATE_LABELS = {
#     0: "Casual browser",
#     1: "Comparison shopper",
#     2: "Purchase-intent",
#     3: "Cold-start / churn",
# }
STATE_LABELS = auto_labels

df_valid["state_label"] = df_valid["hmm_state"].map(STATE_LABELS)
df_users["dominant_label"] = df_users["dominant_state"].map(STATE_LABELS)

## 10. GMM vs HMM Comparison
> Core comparative analysis for your thesis. Answers Research Question 2.

In [ ]:
if HAS_GMM_LABELS:
    # Merge HMM labels onto the GMM-labelled frame
    df_compare = df_valid[["user_id", "session_id", "hmm_state"]].merge(
        df_raw[["session_id", "gmm_regime"]],
        on="session_id",
        how="inner"
    ).dropna(subset=["gmm_regime", "hmm_state"])

    ari = adjusted_rand_score(df_compare["gmm_regime"].astype(int),
                              df_compare["hmm_state"].astype(int))
    nmi = normalized_mutual_info_score(df_compare["gmm_regime"].astype(int),
                                       df_compare["hmm_state"].astype(int))

    print("=" * 60)
    print("GMM vs HMM Agreement")
    print("=" * 60)
    print(f"  Adjusted Rand Index (ARI) : {ari:.4f}")
    print(f"  Normalized Mutual Info    : {nmi:.4f}")
    print(f"  Interpretation:")
    if ari > 0.7:
        print("    HIGH agreement — both models capture the same core structure.")
        print("    Temporal ordering (HMM) does not dramatically change assignments.")
    elif ari > 0.4:
        print("    MODERATE agreement — HMM reveals additional temporal structure")
        print("    not captured by static GMM clustering.")
    else:
        print("    LOW agreement — HMM and GMM capture fundamentally different")
        print("    behavioral patterns. Temporal dynamics matter significantly.")

    # Cross-tabulation: GMM regime vs HMM state
    crosstab = pd.crosstab(
        df_compare["gmm_regime"].astype(int),
        df_compare["hmm_state"].astype(int),
        normalize="index"
    ).round(3)
    crosstab.index   = [f"GMM R{r}" for r in crosstab.index]
    crosstab.columns = [f"HMM S{s}" for s in crosstab.columns]

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(crosstab, annot=True, fmt=".2f", cmap="Blues", ax=ax,
                linewidths=0.5,
                cbar_kws={"label": "Fraction of sessions"})
    ax.set_xlabel("HMM state")
    ax.set_ylabel("GMM regime")
    ax.set_title(
        f"GMM regime → HMM state mapping  |  ARI={ari:.3f}  NMI={nmi:.3f}",
        fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(f"{CFG['output_dir']}/07_gmm_vs_hmm_crosstab.png", bbox_inches="tight")
    plt.show()
    print("[SAVED] 07_gmm_vs_hmm_crosstab.png")

else:
    print("[INFO] GMM labels not available. Run GMM notebook first to compare.")
    ari, nmi = None, None

## 11. Short-term vs Long-term Preference Analysis
> The key behavioral insight of your thesis. Users are classified as **preference-driven** (stable HMM state = long-term preference dominates) or **session-driven** (high volatility = short-term sessions dominate).

In [ ]:
# ── Classify users into preference types ─────────────────────────────────────
# Threshold: median volatility splits users into two camps
vol_median = df_users["volatility"].median()
ent_median = df_users["state_entropy"].median()

df_users["behavior_type"] = np.where(
    (df_users["volatility"] <= vol_median) & (df_users["is_persistent"]),
    "Preference-driven",
    np.where(
        df_users["volatility"] > vol_median,
        "Session-driven",
        "Mixed"
    )
)

type_counts = df_users["behavior_type"].value_counts()
print("User behavior types:")
for t, c in type_counts.items():
    print(f"  {t:25s}: {c:,} users ({100*c/len(df_users):.1f}%)")

In [ ]:
# ── Compare behavioral types ──────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

type_palette = {"Preference-driven": PALETTE[0],
                "Session-driven":     PALETTE[1],
                "Mixed":              PALETTE[2]}

# 1. Volatility distribution
ax1 = fig.add_subplot(gs[0, 0])
for btype, color in type_palette.items():
    sub = df_users[df_users["behavior_type"] == btype]["volatility"]
    ax1.hist(sub, bins=30, alpha=0.6, color=color, label=btype, edgecolor="none")
ax1.set_xlabel("Volatility (transitions / session length)")
ax1.set_ylabel("Users")
ax1.set_title("Volatility distribution", fontweight="bold")
ax1.legend(fontsize=8)

# 2. State entropy distribution
ax2 = fig.add_subplot(gs[0, 1])
for btype, color in type_palette.items():
    sub = df_users[df_users["behavior_type"] == btype]["state_entropy"]
    ax2.hist(sub, bins=30, alpha=0.6, color=color, label=btype, edgecolor="none")
ax2.set_xlabel("State entropy (behavioral diversity)")
ax2.set_title("State entropy distribution", fontweight="bold")
ax2.legend(fontsize=8)

# 3. Behavior type pie
ax3 = fig.add_subplot(gs[0, 2])
ax3.pie(
    type_counts.values,
    labels=type_counts.index,
    colors=[type_palette.get(t, PALETTE[3]) for t in type_counts.index],
    autopct="%1.1f%%", startangle=90,
    pctdistance=0.8, textprops={"fontsize": 10}
)
ax3.set_title("User behavior type distribution", fontweight="bold")

# 4. Volatility vs state entropy scatter
ax4 = fig.add_subplot(gs[1, 0])
for btype, color in type_palette.items():
    sub = df_users[df_users["behavior_type"] == btype]
    ax4.scatter(sub["volatility"], sub["state_entropy"],
                alpha=0.4, s=10, color=color, label=btype)
ax4.set_xlabel("Volatility")
ax4.set_ylabel("State entropy")
ax4.set_title("Volatility vs entropy (user map)", fontweight="bold")
ax4.legend(fontsize=8, markerscale=3)

# 5. N-transitions histogram
ax5 = fig.add_subplot(gs[1, 1])
for btype, color in type_palette.items():
    sub = df_users[df_users["behavior_type"] == btype]["n_transitions"]
    ax5.hist(sub, bins=30, alpha=0.6, color=color, label=btype, edgecolor="none")
ax5.set_xlabel("Number of state transitions")
ax5.set_title("State transitions per user", fontweight="bold")
ax5.legend(fontsize=8)

# 6. Sessions per user by behavior type
ax6 = fig.add_subplot(gs[1, 2])
data_by_type = [
    df_users[df_users["behavior_type"] == btype]["n_sessions"].values
    for btype in type_counts.index
]
bp = ax6.boxplot(data_by_type, patch_artist=True, showfliers=False,
                 medianprops=dict(color="black", lw=2))
for patch, btype in zip(bp["boxes"], type_counts.index):
    patch.set_facecolor(type_palette.get(btype, PALETTE[3]))
    patch.set_alpha(0.7)
ax6.set_xticklabels(type_counts.index, rotation=15, ha="right", fontsize=9)
ax6.set_ylabel("Sessions per user")
ax6.set_title("Engagement level by behavior type", fontweight="bold")

fig.suptitle("Short-term (session-driven) vs Long-term (preference-driven) behavior",
             fontsize=14, fontweight="bold")
plt.savefig(f"{CFG['output_dir']}/08_short_vs_long_term.png", bbox_inches="tight")
plt.show()
print("[SAVED] 08_short_vs_long_term.png")

In [ ]:
# ── Statistical comparison of behavior types ──────────────────────────────────
from scipy import stats

compare_metrics = ["n_sessions", "volatility", "state_entropy", "n_transitions"]

pref_users = df_users[df_users["behavior_type"] == "Preference-driven"]
sess_users = df_users[df_users["behavior_type"] == "Session-driven"]

if len(pref_users) > 0 and len(sess_users) > 0:
    print("\nKruskal-Wallis test: Preference-driven vs Session-driven users")
    print("-" * 65)
    for metric in compare_metrics:
        a = pref_users[metric].dropna().values
        b = sess_users[metric].dropna().values
        stat, p = stats.kruskal(a, b)
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        print(f"  {metric:30s}: H={stat:.2f}  p={p:.4f}  {sig}")
        print(f"    Preference-driven mean : {np.mean(a):.3f}")
        print(f"    Session-driven mean    : {np.mean(b):.3f}")

## 12. Dimensionality Reduction Visualizations

In [ ]:
# ── PCA ───────────────────────────────────────────────────────────────────────
pca  = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)

plot_n    = min(30_000, len(df_valid))
plot_idx  = np.random.choice(len(df_valid), plot_n, replace=False)
plot_labels = viterbi_labels[plot_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for k in range(BEST_K):
    mask = plot_labels == k
    ax.scatter(X_pca[plot_idx][mask, 0], X_pca[plot_idx][mask, 1],
               s=5, alpha=0.4, color=PALETTE[k],
               label=f"S{k}: {STATE_LABELS.get(k, k)}")
    # Emission means in PCA space
    m2d = pca.transform(final_hmm.means_[k:k+1])
    ax.scatter(m2d[0, 0], m2d[0, 1], s=250, marker="X",
               color=PALETTE[k], edgecolor="black", lw=1.5, zorder=10)

ax.set_title("PCA — HMM Viterbi state assignments", fontweight="bold")
ax.set_xlabel(f"PC1 ({100*pca.explained_variance_ratio_[0]:.1f}% var)")
ax.set_ylabel(f"PC2 ({100*pca.explained_variance_ratio_[1]:.1f}% var)")
ax.legend(markerscale=3, fontsize=9)

ax = axes[1]
sc = ax.scatter(
    X_pca[plot_idx, 0], X_pca[plot_idx, 1],
    c=posteriors[plot_idx].max(axis=1),
    cmap="RdYlGn", s=4, alpha=0.5, vmin=0.3, vmax=1.0
)
plt.colorbar(sc, ax=ax, label="Max posterior probability")
ax.set_title("PCA — Viterbi assignment confidence", fontweight="bold")
ax.set_xlabel(f"PC1 ({100*pca.explained_variance_ratio_[0]:.1f}% var)")
ax.set_ylabel(f"PC2 ({100*pca.explained_variance_ratio_[1]:.1f}% var)")

plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/09_pca_visualization.png", bbox_inches="tight")
plt.show()
print("[SAVED] 09_pca_visualization.png")

In [ ]:
# ── UMAP ──────────────────────────────────────────────────────────────────────
if UMAP_AVAILABLE:
    print("Running UMAP...")
    umap_n  = min(50_000, len(df_valid))
    umap_idx = np.random.choice(len(df_valid), umap_n, replace=False)
    reducer  = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                         random_state=RANDOM_STATE, verbose=False)
    X_umap = reducer.fit_transform(X[umap_idx])

    fig, ax = plt.subplots(figsize=(9, 7))
    for k in range(BEST_K):
        mask = viterbi_labels[umap_idx] == k
        ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
                   s=5, alpha=0.4, color=PALETTE[k],
                   label=f"S{k}: {STATE_LABELS.get(k, k)}")
    ax.set_title("UMAP — HMM state assignments", fontweight="bold", fontsize=12)
    ax.set_xlabel("UMAP dim 1")
    ax.set_ylabel("UMAP dim 2")
    ax.legend(markerscale=4, fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{CFG['output_dir']}/10_umap_visualization.png", bbox_inches="tight")
    plt.show()
    print("[SAVED] 10_umap_visualization.png")
else:
    print("[SKIPPED] UMAP not installed.")

## 13. Stability Validation

In [ ]:
def bootstrap_hmm_stability(
    X: np.ndarray,
    lengths: list,
    k: int,
    cov_type: str,
    n_bootstrap: int,
    sample_frac: float,
    random_state: int,
) -> dict:
    """
    Bootstrap stability for HMM via user-level resampling.

    Unlike GMM (which can resample observations freely), HMM
    requires contiguous sequences. We resample USERS (not sessions)
    to preserve the sequence structure that the Markov model depends on.

    Returns dict with mean/std ARI across bootstrap runs.
    """
    rng = np.random.default_rng(random_state)
    n_users = len(lengths)
    n_sample = int(n_users * sample_frac)

    # Build per-user index arrays
    user_starts = np.concatenate([[0], np.cumsum(lengths[:-1])])
    user_ends   = np.cumsum(lengths)

    all_labels = []

    for i in tqdm(range(n_bootstrap), desc="HMM bootstrap stability"):
        user_idx = rng.choice(n_users, n_sample, replace=True)

        # Reconstruct X_boot and lengths_boot for sampled users
        X_chunks  = [X[user_starts[u]:user_ends[u]] for u in user_idx]
        X_boot    = np.vstack(X_chunks)
        len_boot  = [lengths[u] for u in user_idx]

        model, _ = fit_hmm_safe(
            X_boot, len_boot, k, cov_type,
            n_init=3, max_iter=100, tol=1e-3,
            random_state=int(rng.integers(0, 10_000))
        )
        if model is None:
            continue

        # Predict on full dataset with bootstrap model
        all_labels.append(model.predict(X, lengths))

    ari_scores = [
        adjusted_rand_score(all_labels[i], all_labels[j])
        for i in range(len(all_labels))
        for j in range(i + 1, len(all_labels))
    ]

    return {
        "mean_ari": np.mean(ari_scores),
        "std_ari":  np.std(ari_scores),
        "min_ari":  np.min(ari_scores),
        "all_ari":  ari_scores,
        "n_runs":   len(all_labels),
    }


print(f"Running HMM bootstrap stability ({CFG['bootstrap_n']} runs, user-level resampling)...")
stability = bootstrap_hmm_stability(
    X=X, lengths=lengths, k=BEST_K,
    cov_type=CFG["final_cov_type"],
    n_bootstrap=CFG["bootstrap_n"],
    sample_frac=CFG["bootstrap_sample_frac"],
    random_state=RANDOM_STATE,
)

print(f"\n── Stability Results ──────────────────────────────")
print(f"  Successful runs : {stability['n_runs']}")
print(f"  Mean ARI        : {stability['mean_ari']:.4f}")
print(f"  Std  ARI        : {stability['std_ari']:.4f}")
print(f"  Min  ARI        : {stability['min_ari']:.4f}")

if stability["mean_ari"] > 0.85:
    print("  ✅ HIGH stability")
elif stability["mean_ari"] > 0.60:
    print("  ⚠️  MODERATE stability")
else:
    print("  ❌ LOW stability — consider different K or features")

In [ ]:
# ── Plot stability ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(stability["all_ari"], bins=20, color=PALETTE[3], alpha=0.75, edgecolor="none")
ax.axvline(stability["mean_ari"], color="red", lw=2, ls="--",
           label=f"Mean ARI = {stability['mean_ari']:.3f}")
ax.axvline(0.85, color="green", lw=1.5, ls=":", label="Threshold (0.85)")
ax.set_xlabel("Adjusted Rand Index (pairwise bootstrap)")
ax.set_ylabel("Count")
ax.set_title(f"HMM bootstrap stability — K={BEST_K} (n={stability['n_runs']} runs)",
             fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/11_stability_validation.png", bbox_inches="tight")
plt.show()
print("[SAVED] 11_stability_validation.png")

## 14. Saving Outputs & Run Summary

In [ ]:
# ── Save labelled session dataset ─────────────────────────────────────────────
output_cols = (
    ["user_id", "session_id", "hmm_state", "state_label", "max_posterior"]
    + [f"prob_state_{k}" for k in range(BEST_K)]
    + CFG["features"]
)
if HAS_TRUE_LABELS:
    output_cols.insert(3, "true_regime")
if HAS_GMM_LABELS and "gmm_regime" in df_valid.columns:
    output_cols.insert(4, "gmm_regime")
if CFG.get("time_col") and CFG["time_col"] in df_valid.columns:
    output_cols.insert(2, CFG["time_col"])

df_out = df_valid[[c for c in output_cols if c in df_valid.columns]]
df_out.to_parquet(f"{CFG['output_dir']}/sessions_with_hmm_labels.parquet", index=False)
df_out.to_csv(f"{CFG['output_dir']}/sessions_with_hmm_labels.csv", index=False)
print(f"[SAVED] sessions_with_hmm_labels (.parquet + .csv) — {len(df_out):,} rows")

# ── Save per-user state summary ───────────────────────────────────────────────
df_users.to_parquet(f"{CFG['output_dir']}/users_state_summary.parquet", index=False)
df_users.to_csv(f"{CFG['output_dir']}/users_state_summary.csv", index=False)
print(f"[SAVED] users_state_summary (.parquet + .csv) — {len(df_users):,} users")

# ── Save transition matrix ────────────────────────────────────────────────────
pd.DataFrame(
    trans_matrix,
    index=[f"From State {k}" for k in range(BEST_K)],
    columns=[f"To State {k}" for k in range(BEST_K)]
).to_csv(f"{CFG['output_dir']}/transition_matrix.csv")
print("[SAVED] transition_matrix.csv")

In [ ]:
# ── Validate against true labels (simulation only) ────────────────────────────
if HAS_TRUE_LABELS:
    ari_true  = adjusted_rand_score(df_valid["true_regime"], df_valid["hmm_state"])
    nmi_true  = normalized_mutual_info_score(df_valid["true_regime"], df_valid["hmm_state"])
    print(f"Recovery vs true regimes — ARI: {ari_true:.4f}  NMI: {nmi_true:.4f}")

    cm = confusion_matrix(df_valid["true_regime"], df_valid["hmm_state"])
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=[f"HMM {k}" for k in range(BEST_K)],
                yticklabels=[f"True {k}" for k in range(len(cm))])
    ax.set_title("Confusion matrix: true vs HMM state", fontweight="bold")
    ax.set_xlabel("Predicted state")
    ax.set_ylabel("True regime")
    plt.tight_layout()
    plt.savefig(f"{CFG['output_dir']}/12_confusion_matrix_true.png", bbox_inches="tight")
    plt.show()
    print("[SAVED] 12_confusion_matrix_true.png")

In [ ]:
# ── Full run summary JSON ─────────────────────────────────────────────────────
summary = {
    "run_timestamp":          datetime.now().isoformat(),
    "model":                  "GaussianHMM",
    "n_sessions":             int(len(df_valid)),
    "n_users":                int(len(sequences)),
    "features_used":          CFG["features"],
    "scaler":                 CFG["scaler"],
    "log_transformed":        CFG["log_transform_cols"],
    "best_k":                 BEST_K,
    "covariance_type":        CFG["final_cov_type"],
    "log_likelihood_per_obs": float(final_score),
    "bic":                    float(hmm_bic(final_hmm, X, lengths)),
    "converged":              bool(final_hmm.monitor_.converged),
    "stability_mean_ari":     float(stability["mean_ari"]),
    "stability_std_ari":      float(stability["std_ari"]),
    "state_labels":           {str(k): v for k, v in STATE_LABELS.items()},
    "state_sizes_sessions":   {str(k): int((viterbi_labels == k).sum()) for k in range(BEST_K)},
    "transition_matrix":      trans_matrix.tolist(),
    "stationary_distribution":stationary.tolist(),
    "behavior_type_counts":   df_users["behavior_type"].value_counts().to_dict(),
    "pct_preference_driven":  float((df_users["behavior_type"] == "Preference-driven").mean()),
    "pct_session_driven":     float((df_users["behavior_type"] == "Session-driven").mean()),
    "high_confidence_pct":    float((posteriors.max(axis=1) > 0.8).mean()),
}

if HAS_TRUE_LABELS:
    summary["recovery_ari"] = float(ari_true)
    summary["recovery_nmi"] = float(nmi_true)
if ari is not None:
    summary["gmm_vs_hmm_ari"] = float(ari)
    summary["gmm_vs_hmm_nmi"] = float(nmi)

with open(f"{CFG['output_dir']}/run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("[SAVED] run_summary.json")
print("\n" + "=" * 65)
print("FINAL SUMMARY — GaussianHMM")
print("=" * 65)
skip = {"features_used", "log_transformed", "transition_matrix", "stationary_distribution"}
for k, v in summary.items():
    if k not in skip:
        print(f"  {k:40s}: {v}")
print("=" * 65)
print(f"\nAll outputs saved to: ./{CFG['output_dir']}/")

---
## Thesis Write-up Notes

### Key results to report from this notebook

| Finding | Where to look |
|---|---|
| Optimal K and selection criteria | Section 5 — BIC/AIC curves |
| State emission profiles | Section 9 — radar chart + profile_df |
| Transition dynamics | Section 8 — transition matrix heatmap + MFPT |
| Example user trajectories | Section 7 — trajectory plots |
| GMM vs HMM agreement | Section 10 — ARI, NMI, crosstab |
| Short vs long-term split | Section 11 — behavior_type analysis |
| Model stability | Section 13 — bootstrap ARI |

### Research Question 2 answer template
> *"The HMM analysis identified K={BEST_K} latent behavioral states. Approximately X% of users exhibited preference-driven behavior (low volatility, >70% of sessions in one state), while Y% were session-driven (high transition frequency, high state entropy). The GMM–HMM agreement ARI of Z indicates that [temporal dynamics add/do not add] substantial explanatory power beyond static clustering. Users in the purchase-intent state required on average M sessions (MFPT) to reach from the casual browsing state, with a self-transition probability of P — meaning purchase intent is [persistent / transient]."*

---
*End of HMM main model notebook*